[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qualia906/Developing-Agentic-AI-with-LangChain/blob/main/chap05/hands-on/chap05_hands_on.ipynb)


# 第5章 ハンズオン：マルチエージェントシステムの構築（Supervisor パターン）

## 学習目標
- 複数のサブエージェントを `create_agent` で作成する
- サブエージェントを `@tool` でラップし、Supervisor エージェントから呼び出せるようにする
- Supervisor エージェントがサブエージェントを協調させるマルチエージェントシステムを構築する
- LangSmith で階層的なトレースを確認する

---

In [ ]:
%pip install -qU langchain langchain-openai langgraph

In [ ]:
import os
from google.colab import userdata

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = "chap05-handson"
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

print("環境変数の設定が完了しました")

## Part 1: サブエージェントの作成

今回は **コンテンツ生成チーム** を想定したマルチエージェントシステムを構築します。

```
【システム構成】
Supervisor Agent
  ├── research_agent  : トピックのリサーチを担当
  ├── writer_agent    : 記事の執筆を担当
  └── validator_agent : 内容のチェックを担当
```

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool

# ─── サブエージェント1: リサーチエージェント ───
# 指定されたトピックについて情報を収集・整理する専門エージェント
research_agent = create_agent(
    model="openai:gpt-4o",
    tools=[],   # このデモでは LLM の知識のみ使用
    system_prompt=(
        "あなたはリサーチの専門家です。\n"
        "与えられたトピックについて、重要なポイントを3〜5つ箇条書きでまとめてください。\n"
        "事実に基づいた正確な情報を提供してください。"
    ),
)

# ─── サブエージェント2: ライターエージェント ───
# リサーチ結果をもとに読みやすい記事を執筆する専門エージェント
writer_agent = create_agent(
    model="openai:gpt-4o",
    tools=[],
    system_prompt=(
        "あなたはテクノロジー分野のプロのライターです。\n"
        "提供されたリサーチ情報をもとに、一般のビジネスパーソン向けの\n"
        "わかりやすい記事（300〜500文字）を執筆してください。\n"
        "見出し、本文の構成を工夫して、読みやすい文章にしてください。"
    ),
)

# ─── サブエージェント3: バリデーターエージェント ───
# 執筆された記事の品質チェックを行う専門エージェント
validator_agent = create_agent(
    model="openai:gpt-4o",
    tools=[],
    system_prompt=(
        "あなたはコンテンツレビューの専門家です。\n"
        "提供された記事を以下の観点でレビューし、スコアとフィードバックを提供してください:\n"
        "1. 正確性（情報は正しいか）: /10\n"
        "2. 読みやすさ（文章は明瞭か）: /10\n"
        "3. 完成度（構成・内容は十分か）: /10\n"
        "最後に総合スコアと改善提案を述べてください。"
    ),
)

print("3つのサブエージェントの作成が完了しました")

## Part 2: サブエージェントをツールとしてラップ

Supervisor エージェントからサブエージェントを呼び出せるように、
各サブエージェントを `@tool` でラップします。

これが Supervisor パターンの核心です：
**エージェントがエージェントをツールとして呼び出す**

In [ ]:
@tool
def research(topic: str) -> str:
    """指定されたトピックについてリサーチを行い、重要なポイントをまとめます。
    
    Args:
        topic: リサーチするトピック（例: '生成AIの最新トレンド'）
    """
    # サブエージェントに委譲
    result = research_agent.invoke({
        "messages": [
            {"role": "user", "content": f"以下のトピックについてリサーチしてください：{topic}"}
        ]
    })
    # サブエージェントの最終回答を文字列として返す
    return result["messages"][-1].content


@tool
def write_article(research_result: str, topic: str) -> str:
    """リサーチ結果をもとに、指定トピックの記事を執筆します。
    
    Args:
        research_result: リサーチエージェントが出力した調査結果
        topic: 記事のトピック
    """
    result = writer_agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": (
                    f"トピック: {topic}\n\n"
                    f"リサーチ結果:\n{research_result}\n\n"
                    "上記のリサーチ結果をもとに記事を執筆してください。"
                )
            }
        ]
    })
    return result["messages"][-1].content


@tool
def validate_article(article: str) -> str:
    """執筆された記事の品質をチェックし、スコアとフィードバックを返します。
    
    Args:
        article: チェックする記事の本文
    """
    result = validator_agent.invoke({
        "messages": [
            {"role": "user", "content": f"以下の記事をレビューしてください：\n\n{article}"}
        ]
    })
    return result["messages"][-1].content


print("サブエージェントのツールラップが完了しました")

## Part 3: Supervisor エージェントの作成と実行

In [ ]:
# Supervisor エージェント
# リサーチ → 執筆 → バリデーション のフローを自律的に管理する
supervisor_agent = create_agent(
    model="openai:gpt-4o",
    tools=[research, write_article, validate_article],
    system_prompt=(
        "あなたはコンテンツ制作チームのスーパーバイザーです。\n"
        "ユーザーからトピックが与えられたら、以下のステップで記事制作を管理してください：\n"
        "1. research ツールでトピックをリサーチする\n"
        "2. write_article ツールでリサーチ結果をもとに記事を執筆する\n"
        "3. validate_article ツールで記事の品質をチェックする\n"
        "4. 最終的な記事と品質レポートをユーザーに提示する\n"
        "日本語で回答してください。"
    ),
)

print("Supervisor エージェントの作成が完了しました")
print("マルチエージェントシステムを起動します...")
print()

In [ ]:
# Supervisor エージェントを実行
# 「LangChain の活用事例」というトピックで記事を作成させる
result = supervisor_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "『LangChain を使った企業の AI 活用事例』というトピックで記事を作成してください"
        }
    ]
})

print("=" * 60)
print("Supervisor エージェントの最終出力:")
print("=" * 60)
print(result["messages"][-1].content)

In [ ]:
# メッセージフローを確認
# サブエージェント呼び出しがトレースに記録されていることを確認する
print("=== 実行されたメッセージフロー ===")
print(f"総メッセージ数: {len(result['messages'])}")
print()

for i, msg in enumerate(result["messages"]):
    msg_type = type(msg).__name__
    print(f"[{i+1}] {msg_type}", end=" ")
    
    # ツール呼び出し（AIMessage → ToolMessage の流れ）
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        tool_names = [tc['name'] for tc in msg.tool_calls]
        print(f"→ ツール呼び出し: {tool_names}")
    elif msg_type == "ToolMessage":
        print(f"→ ToolMessage [{msg.name}] ({len(msg.content)}文字)")
    else:
        preview = msg.content[:60] + "..." if len(msg.content) > 60 else msg.content
        print(f"| {preview}")

print()
print("LangSmith（https://smith.langchain.com/）でトレースを確認してみましょう！")

---

## まとめ

| 概念 | 実装方法 | ポイント |
|------|---------|--------|
| サブエージェントの作成 | `create_agent(model=..., tools=[], system_prompt=...)` | 役割を明確に定義したシステムプロンプトが重要 |
| ツールとしてのラップ | `@tool` デコレータ + `sub_agent.invoke(...)` | エージェントがエージェントをツールとして呼び出す |
| Supervisor パターン | Supervisor に orchestration ロジックをシステムプロンプトで指示 | LLM の判断力を活用してフローを自律管理 |

**次のステップ:** `chap05/app/app.py` でこのマルチエージェントシステムを
Streamlit Web アプリとして UI 化します。